In [1]:
import os
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc
from dotenv import load_dotenv

load_dotenv()

Total rows in code: 170974447
Total rows in new_low_medium_quality: 17935258
Total rows in english_low_medium_quality: 93036106
Total rows in dutch_low_medium_quality: 7612219
Total rows in english_high_quality: 3443599
Total rows in other_languages: 3297160
Total rows in new_high_quality: 62306664
Total rows in dutch_high_quality: 5893233


# Tagged Epoch 2 data (update 02/08/2025)

```
Total rows processed: 364557686

code: 171033447 rows
new_high_quality: 62306664 rows
new_low_medium_quality: 17935258 rows
dutch_high_quality: 5893233 rows
dutch_low_medium_quality: 7612219 rows
english_high_quality: 3443599 rows
english_low_medium_quality: 93036106 rows
other_languages: 3297160 rows

# Matches exactly with the logs
Current bucket counts: {
    'new_high_quality': 62306664, 
    'new_low_medium_quality': 17935258, 
    'dutch_high_quality': 5893233, 
    'dutch_low_medium_quality': 7612219, 
    'english_high_quality': 3443599, 
    'english_low_medium_quality': 93036106, 
    'other_languages': 3297160, 
    'code': 171033447}
```

In [8]:
import copy

tagging_output_path = os.path.join(os.environ["GPTNL_2_BUCKETED"], "tagging_output")
original_data_path = os.path.join(os.environ["GPTNL_2_BUCKETED"], "gpt_nl_dataset_v2.0")

columns = [
    "is_code",
    "dataset_quality",
    "is_new_epoch_2",
    "original_file_path",
    "dataset_name",
    "language",
]

datasets = [
    d
    for d in os.listdir(tagging_output_path)
    if os.path.isdir(os.path.join(tagging_output_path, d))
]

doc_counts = {
    "code": 0,
    "new_high_quality": 0,
    "new_low_medium_quality": 0,
    "dutch_high_quality": 0,
    "dutch_low_medium_quality": 0,
    "english_high_quality": 0,
    "english_low_medium_quality": 0,
    "other_languages": 0,
}

doc_counts_per_dataset = {dataset: copy.deepcopy(doc_counts) for dataset in datasets}

for i, dataset in enumerate(datasets):
    print("Processing dataset:", dataset, f"{i + 1}/{len(datasets)}")
    dataset_path = os.path.join(tagging_output_path, dataset)
    for pqt_file in os.listdir(dataset_path):
        pqt_file_path = os.path.join(dataset_path, pqt_file)
        if not pqt_file.endswith(".parquet"):
            continue

        # Read only the specified columns from the parquet file
        table = pq.read_table(pqt_file_path, columns=columns)
        total_rows = table.num_rows

        # Count code rows
        code_mask = pc.equal(table["is_code"], True)
        count_code = pc.sum(pc.cast(code_mask, pa.int64())).as_py()
        doc_counts["code"] += count_code
        doc_counts_per_dataset[dataset]["code"] += count_code

        # Count new high quality rows
        new_high_quality_mask = pc.and_(
            pc.and_(
                pc.equal(table["dataset_quality"], "high"),
                pc.equal(table["is_code"], False),
            ),
            pc.equal(table["is_new_epoch_2"], True),
        )
        count_new_high_quality = pc.sum(
            pc.cast(new_high_quality_mask, pa.int64())
        ).as_py()
        doc_counts["new_high_quality"] += count_new_high_quality
        doc_counts_per_dataset[dataset]["new_high_quality"] += count_new_high_quality

        # Count new low/medium quality rows
        new_low_medium_quality_mask = pc.and_(
            pc.and_(
                pc.is_in(table["dataset_quality"], pa.array(["low", "medium"])),
                pc.equal(table["is_code"], False),
            ),
            pc.equal(table["is_new_epoch_2"], True),
        )
        count_new_low_medium_quality = pc.sum(
            pc.cast(new_low_medium_quality_mask, pa.int64())
        ).as_py()
        doc_counts["new_low_medium_quality"] += count_new_low_medium_quality
        doc_counts_per_dataset[dataset]["new_low_medium_quality"] += (
            count_new_low_medium_quality
        )

        # Count Dutch high quality rows
        dutch_high_quality_mask = pc.and_(
            pc.and_(
                pc.and_(
                    pc.equal(table["dataset_quality"], "high"),
                    pc.equal(table["is_code"], False),
                ),
                pc.equal(table["is_new_epoch_2"], False),
            ),
            pc.equal(table["language"], "nl"),
        )
        count_dutch_high_quality = pc.sum(
            pc.cast(dutch_high_quality_mask, pa.int64())
        ).as_py()
        doc_counts["dutch_high_quality"] += count_dutch_high_quality
        doc_counts_per_dataset[dataset]["dutch_high_quality"] += (
            count_dutch_high_quality
        )

        # Count Dutch low/medium quality rows
        dutch_low_medium_quality_mask = pc.and_(
            pc.and_(
                pc.and_(
                    pc.is_in(table["dataset_quality"], pa.array(["low", "medium"])),
                    pc.equal(table["is_code"], False),
                ),
                pc.equal(table["is_new_epoch_2"], False),
            ),
            pc.equal(table["language"], "nl"),
        )
        count_dutch_low_medium_quality = pc.sum(
            pc.cast(dutch_low_medium_quality_mask, pa.int64())
        ).as_py()
        doc_counts["dutch_low_medium_quality"] += count_dutch_low_medium_quality
        doc_counts_per_dataset[dataset]["dutch_low_medium_quality"] += (
            count_dutch_low_medium_quality
        )

        # Count English high quality rows
        english_high_quality_mask = pc.and_(
            pc.and_(
                pc.and_(
                    pc.equal(table["dataset_quality"], "high"),
                    pc.equal(table["is_code"], False),
                ),
                pc.equal(table["is_new_epoch_2"], False),
            ),
            pc.equal(table["language"], "en"),
        )
        count_english_high_quality = pc.sum(
            pc.cast(english_high_quality_mask, pa.int64())
        ).as_py()
        doc_counts["english_high_quality"] += count_english_high_quality
        doc_counts_per_dataset[dataset]["english_high_quality"] += (
            count_english_high_quality
        )

        # Count English low/medium quality rows
        english_low_medium_quality_mask = pc.and_(
            pc.and_(
                pc.and_(
                    pc.is_in(table["dataset_quality"], pa.array(["low", "medium"])),
                    pc.equal(table["is_code"], False),
                ),
                pc.equal(table["is_new_epoch_2"], False),
            ),
            pc.equal(table["language"], "en"),
        )
        count_english_low_medium_quality = pc.sum(
            pc.cast(english_low_medium_quality_mask, pa.int64())
        ).as_py()
        doc_counts["english_low_medium_quality"] += count_english_low_medium_quality
        doc_counts_per_dataset[dataset]["english_low_medium_quality"] += (
            count_english_low_medium_quality
        )

        # Count other languages rows
        other_languages_mask = pc.and_(
            pc.equal(table["is_code"], False),
            pc.and_(
                pc.invert(pc.is_in(table["language"], pa.array(["nl", "en"]))),
                pc.equal(table["is_new_epoch_2"], False),
            ),
        )

        count_other_languages = pc.sum(
            pc.cast(other_languages_mask, pa.int64())
        ).as_py()
        doc_counts["other_languages"] += count_other_languages
        doc_counts_per_dataset[dataset]["other_languages"] += count_other_languages

missing_count = 0
missing_files = []

for original_data_dir in os.listdir(original_data_path):
    original_data_dir_path = os.path.join(original_data_path, original_data_dir)

    if not os.path.isdir(original_data_dir_path):
        continue

    for pqt_file in os.listdir(original_data_dir_path):
        pqt_file_path = os.path.join(original_data_dir_path, pqt_file)
        new_pqt_file_path = os.path.join(
            tagging_output_path, original_data_dir, pqt_file
        )

        if not os.path.isfile(pqt_file_path):
            missing_count += 1
            missing_files.append(pqt_file_path)
            continue

total_rows = sum(doc_counts.values())

print(f"Total rows processed: {total_rows}")

print()
for key, value in doc_counts.items():
    print(f"{key}: {value} rows")

print()
for key, value in doc_counts.items():
    print(f"{key}: {value / total_rows * 100:.2f} %")

Total rows processed: 364557686

code: 171033447 rows
new_high_quality: 62306664 rows
new_low_medium_quality: 17935258 rows
dutch_high_quality: 5893233 rows
dutch_low_medium_quality: 7612219 rows
english_high_quality: 3443599 rows
english_low_medium_quality: 93036106 rows
other_languages: 3297160 rows

code: 46.92 %
new_high_quality: 17.09 %
new_low_medium_quality: 4.92 %
dutch_high_quality: 1.62 %
dutch_low_medium_quality: 2.09 %
english_high_quality: 0.94 %
english_low_medium_quality: 25.52 %
other_languages: 0.90 %


In [6]:
import pandas as pd

pd.DataFrame.from_dict(doc_counts_per_dataset, orient="index")

In [ ]:
# 2025-08-01 14:02:43,036 - INFO - Current bucket counts: {'new_high_quality': 46533196, 'new_low_medium_quality': 98134842, 'dutch_high_quality': 25807814, 'dutch_low_medium_quality': 4934865, 'english_high_quality': 18557, 'english_low_medium_quality': 15529143, 'other_languages': 2565822, 'code': 171033447}

# Old Epoch 1

```
code: 118900984 rows
dutch_high_quality: 4604500 rows
dutch_low_medium_quality: 7612219 rows
english_high_quality: 3442664 rows
english_low_medium_quality: 93036106 rows
other_languages: 3293998 rows

# matches with logs
Current bucket counts: {'dutch_high_quality': 4604500, 'dutch_low_medium_quality': 7612219, 'english_high_quality': 3442664, 'english_low_medium_quality': 93036106, 'other_languages': 3293998, 'code': 118900984}

code: 51.50 %
dutch_high_quality: 1.99 %
dutch_low_medium_quality: 3.30 %
english_high_quality: 1.49 %
english_low_medium_quality: 40.29 %
other_languages: 1.43 %

In [9]:
import pyarrow as pa
import pyarrow.parquet as pq

tagging_output_path = os.path.join(os.environ["GPTNL_1_BUCKETED"], "tagging_output")

columns = [
    "is_code",
    "dataset_quality",
    "original_file_path",
    "dataset_name",
    "language",
]

datasets = [
    d
    for d in os.listdir(tagging_output_path)
    if os.path.isdir(os.path.join(tagging_output_path, d))
]

doc_counts = {
    "code": 0,
    "dutch_high_quality": 0,
    "dutch_low_medium_quality": 0,
    "english_high_quality": 0,
    "english_low_medium_quality": 0,
    "other_languages": 0,
}

for i, dataset in enumerate(datasets):
    print("Processing dataset:", dataset, f"{i + 1}/{len(datasets)}")
    dataset_path = os.path.join(tagging_output_path, dataset)
    for pqt_file in os.listdir(dataset_path):
        pqt_file_path = os.path.join(dataset_path, pqt_file)
        if not pqt_file.endswith(".parquet"):
            continue

        # Read only the specified columns from the parquet file
        table = pq.read_table(pqt_file_path, columns=columns)
        total_rows = table.num_rows

        # Count code rows
        code_mask = pc.equal(table["is_code"], True)
        count_code = pc.sum(pc.cast(code_mask, pa.int64())).as_py()
        doc_counts["code"] += count_code

        # Count Dutch high quality rows
        dutch_high_quality_mask = pc.and_(
            pc.and_(
                pc.equal(table["dataset_quality"], "high"),
                pc.equal(table["is_code"], False),
            ),
            pc.equal(table["language"], "nl"),
        )
        count_dutch_high_quality = pc.sum(
            pc.cast(dutch_high_quality_mask, pa.int64())
        ).as_py()
        doc_counts["dutch_high_quality"] += count_dutch_high_quality

        # Count Dutch low/medium quality rows
        dutch_low_medium_quality_mask = pc.and_(
            pc.and_(
                pc.is_in(table["dataset_quality"], pa.array(["low", "medium"])),
                pc.equal(table["is_code"], False),
            ),
            pc.equal(table["language"], "nl"),
        )
        count_dutch_low_medium_quality = pc.sum(
            pc.cast(dutch_low_medium_quality_mask, pa.int64())
        ).as_py()
        doc_counts["dutch_low_medium_quality"] += count_dutch_low_medium_quality

        # Count English high quality rows
        english_high_quality_mask = pc.and_(
            pc.and_(
                pc.equal(table["dataset_quality"], "high"),
                pc.equal(table["is_code"], False),
            ),
            pc.equal(table["language"], "en"),
        )
        count_english_high_quality = pc.sum(
            pc.cast(english_high_quality_mask, pa.int64())
        ).as_py()
        doc_counts["english_high_quality"] += count_english_high_quality

        # Count English low/medium quality rows
        english_low_medium_quality_mask = pc.and_(
            pc.and_(
                pc.is_in(table["dataset_quality"], pa.array(["low", "medium"])),
                pc.equal(table["is_code"], False),
            ),
            pc.equal(table["language"], "en"),
        )
        count_english_low_medium_quality = pc.sum(
            pc.cast(english_low_medium_quality_mask, pa.int64())
        ).as_py()
        doc_counts["english_low_medium_quality"] += count_english_low_medium_quality

        # Count other languages rows
        other_languages_mask = pc.and_(
            pc.equal(table["is_code"], False),
            pc.invert(pc.is_in(table["language"], pa.array(["nl", "en"]))),
        )

        count_other_languages = pc.sum(
            pc.cast(other_languages_mask, pa.int64())
        ).as_py()
        doc_counts["other_languages"] += count_other_languages

total_rows = sum(doc_counts.values())

print(f"Total rows processed: {total_rows}")

print()
for key, value in doc_counts.items():
    print(f"{key}: {value} rows")

print()
for key, value in doc_counts.items():
    print(f"{key}: {value / total_rows * 100:.2f} %")

Total rows processed: 230890471

code: 118900984 rows
dutch_high_quality: 4604500 rows
dutch_low_medium_quality: 7612219 rows
english_high_quality: 3442664 rows
english_low_medium_quality: 93036106 rows
other_languages: 3293998 rows

code: 51.50 %
dutch_high_quality: 1.99 %
dutch_low_medium_quality: 3.30 %
english_high_quality: 1.49 %
english_low_medium_quality: 40.29 %
other_languages: 1.43 %
